# 34 — OpenAI Agents SDK

Build a triage agent that routes questions to specialist agents using **handoffs** — the OpenAI Agents SDK primitive for agent-to-agent transfer.

**What you'll learn**
- `Agent`: core building block with name, instructions, model, tools, and handoffs
- `@tool`: decorator that wraps any Python function into a callable tool
- `handoff(agent)`: transfers control from triage to a specialist
- `Runner.run_sync()`: synchronous execution returning a `RunResult`
- Built-in tracing: zero-config step recording vs LangSmith
- Contrast with LangGraph: handoffs vs `add_conditional_edges`

**Pattern:** Triage → [Researcher (has tools) | Writer (no tools)]

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q openai-agents python-dotenv

In [ ]:
import os

from dotenv import load_dotenv

if not IN_COLAB:
    load_dotenv()
if IN_COLAB and not os.environ.get("OPENAI_API_KEY"):
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running"

In [ ]:
# ── 3. Agents SDK vs LangGraph — key comparison ───────────────────────────────
#
# LangGraph:        nodes + edges on a StateGraph
#   graph.add_node("researcher", researcher_fn)
#   graph.add_conditional_edges("triage", route, {"researcher": "researcher"})
#
# OpenAI Agents SDK: agents + handoffs (no explicit graph)
#   researcher = Agent(name="Researcher", ...)
#   triage = Agent(name="Triage", handoffs=[handoff(researcher)])
#   Runner.run_sync(triage, question)
#
# Key differences:
#   - Agents SDK: implicit state (conversation history), built-in tracing
#   - LangGraph:  explicit TypedDict state, requires LangSmith for tracing
#   - Agents SDK: simpler for chat-based handoffs
#   - LangGraph:  better for complex state machines and DAGs

print("LangGraph:  StateGraph + add_conditional_edges (explicit graph)")
print("Agents SDK: Agent + handoff() (implicit routing via instructions)")

In [ ]:
# ── 4. Build the triage agent system ─────────────────────────────────────────
from agents import Agent, Runner, handoff, tool

MODEL = "gpt-4o-mini"

DOCS = [
    "OpenAI Agents SDK was released by OpenAI in early 2025.",
    "OpenAI Agents SDK uses Agent, Runner, tool, and handoff as its core primitives.",
    "An Agent is defined with a name, instructions, model, tools, and optional handoffs.",
    "Handoffs transfer control from one agent to another, like routing in LangGraph.",
    "Runner.run_sync() executes an agent synchronously and returns a RunResult.",
    "Built-in tracing in the Agents SDK records every step without external setup.",
    "The @tool decorator wraps a Python function into a tool callable by any Agent.",
    "LangGraph uses nodes and edges to build graphs; Agents SDK uses handoffs between agents.",
    "The Agents SDK supports parallel tool calls and structured output natively.",
    "Agents SDK agents can share tools — a tool defined once can be used by any agent.",
]


@tool
def keyword_search(query: str) -> str:
    """Search the knowledge base for relevant facts about the OpenAI Agents SDK."""
    words = set(query.lower().split())
    scored = [(sum(w in d.lower() for w in words), d) for d in DOCS]
    scored.sort(reverse=True)
    top = [d for _, d in scored[:3] if _ > 0]
    return "\n".join(top) if top else "No relevant facts found."


researcher = Agent(
    name="Researcher",
    model=MODEL,
    instructions=(
        "You answer factual questions about the OpenAI Agents SDK. "
        "Always call keyword_search first to retrieve relevant facts, "
        "then compose a clear 2-3 sentence answer."
    ),
    tools=[keyword_search],
)

writer = Agent(
    name="Writer",
    model=MODEL,
    instructions=(
        "You receive research findings and rewrite them as a polished, "
        "readable paragraph suitable for a technical blog post."
    ),
)

triage = Agent(
    name="Triage",
    model=MODEL,
    instructions=(
        "Route the user's request: "
        "if they ask a factual question, hand off to Researcher. "
        "If they ask to explain, summarize, or write something, hand off to Writer. "
        "Do not answer directly — always hand off."
    ),
    handoffs=[handoff(researcher), handoff(writer)],
)

print("Agents ready: Triage → [Researcher, Writer]")

In [ ]:
# ── 5. Run the triage system ──────────────────────────────────────────────────
QUESTIONS = [
    "What are the core primitives of the OpenAI Agents SDK?",
    "How does the Agents SDK differ from LangGraph?",
    "How does built-in tracing work in the Agents SDK?",
]

for q in QUESTIONS:
    result = Runner.run_sync(triage, q)
    print(f"Q: {q}")
    print(f"A: {result.final_output}")
    print()

## Exercises

**Exercise 1 — Add a third specialist:** Create a `CodeAgent` that generates Python code snippets. Update Triage's instructions to route code requests to it.

**Exercise 2 — Inspect tracing:** After running, call `result.trace` or check the OpenAI dashboard to see the full trace of which agents were called and in what order.

**Exercise 3 — Share a tool:** Give `keyword_search` to both Researcher and Writer. Ask a writing question — does Writer call the search tool before writing?

**Exercise 4 — LangGraph comparison:** Implement the same triage flow in LangGraph with `add_conditional_edges`. Compare lines of code and how routing logic is expressed.

## What's next

- **6-multi-agent-graph** — same multi-agent pattern in LangGraph with router LLM
- **31-multi-agent-debate** — agents that argue against each other before a judge decides
- **11-hitl-approval** — add a human-in-the-loop gate between triage and specialist